# 00 Setup

**Doel:** Eenmalige (maar idempotente) aanmaak van:
- De Unity Catalog `DEMO`
- De drie schema's: `STAGING_AZURESTORAGE`, `STAGING_SQLSERVER`, `CONFIG`
- Het External Volume op de Azure Storage External Location
- De control table `DEMO.CONFIG.pipeline_sources` met twee seed-rijen

Dit notebook is volledig idempotent: meerdere keren draaien geeft hetzelfde resultaat
en leidt niet tot fouten of dubbele rijen.

In [0]:
# Parameters & widgets
dbutils.widgets.text("catalog", "DEMO", "Catalog")
dbutils.widgets.text(
    "parquet_storage_location",
    "",
    "Parquet Storage URL (abfss://...)",
)
dbutils.widgets.text(
    "catalog_managed_location",
    "",
    "Catalog Managed Location URL (abfss://...)",
)

catalog = dbutils.widgets.get("catalog")
parquet_location = dbutils.widgets.get("parquet_storage_location")
managed_location = dbutils.widgets.get("catalog_managed_location")

print(f"Catalog          : {catalog}")
print(f"Parquet Location : {parquet_location}")
print(f"Managed Location : {managed_location}")

## Stap 1 — Catalog aanmaken

In [0]:
spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {catalog}
    MANAGED LOCATION '{managed_location}'
""")
spark.sql(f"USE CATALOG {catalog}")

print(f"Catalog '{catalog}' bestaat of is aangemaakt met managed location '{managed_location}'.")

## Stap 2 — Schema's aanmaken

In [0]:
for schema in ["STAGING_AZURESTORAGE", "STAGING_SQLSERVER", "CONFIG"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    print(f"Schema '{catalog}.{schema}' bestaat of is aangemaakt.")

## Stap 3 — External Volume aanmaken

Het volume wijst naar de `source/` sub-map van de Azure Storage container die wordt
beheerd door de opgegeven External Location.

In [0]:
volume_azure_storage = f"""
CREATE EXTERNAL VOLUME IF NOT EXISTS {catalog}.STAGING_AZURESTORAGE.parquet
  LOCATION '{parquet_location}'
"""

spark.sql(volume_azure_storage)
print(
    f"External Volume '{catalog}.STAGING_AZURESTORAGE.parquet' bestaat of is aangemaakt."
)

## Stap 4 — Control table aanmaken

In [0]:
create_control_table_sql = f"""
CREATE TABLE IF NOT EXISTS {catalog}.CONFIG.pipeline_sources (
  source_system  STRING  NOT NULL COMMENT 'Bronsysteem (bijv. azurestorage)',
  source_path    STRING  NOT NULL COMMENT 'Pad naar de bronfolder (volume-pad)',
  file_pattern   STRING  NOT NULL COMMENT 'Glob filter binnen de folder (per doeltabel)',
  target_schema  STRING  NOT NULL COMMENT 'Doelschema binnen de catalog',
  target_table   STRING  NOT NULL COMMENT 'Doeltabelnaam',
  file_format    STRING  NOT NULL COMMENT 'Bestandstype (bijv. parquet)',
  is_active      BOOLEAN NOT NULL COMMENT 'Aan/uit zonder rij te verwijderen',
  load_type      STRING  NOT NULL COMMENT 'full of incremental (Auto Loader)'
)
USING DELTA
COMMENT 'Control table — stuurt de parquet-ingest pipeline aan'
"""

spark.sql(create_control_table_sql)
print(f"Control table '{catalog}.CONFIG.pipeline_sources' bestaat of is aangemaakt.")

## Stap 5 — Seed-rijen laden (MERGE — idempotent)

In [0]:
source_path = f"/Volumes/{catalog.lower()}/staging_azurestorage/parquet"

merge_sql = f"""
MERGE INTO {catalog}.CONFIG.pipeline_sources AS tgt
USING (
  SELECT
    'azurestorage'                                      AS source_system,
    '{source_path}'                                     AS source_path,
    'ORDER_HEADER*.parquet'                            AS file_pattern,
    'STAGING_AZURESTORAGE'                              AS target_schema,
    'order_header'                                      AS target_table,
    'parquet'                                           AS file_format,
    true                                                AS is_active,
    'full'                                              AS load_type
  UNION ALL
  SELECT
    'azurestorage'                                      AS source_system,
    '{source_path}'                                     AS source_path,
    'ORDER_DETAIL*.parquet'                            AS file_pattern,
    'STAGING_AZURESTORAGE'                              AS target_schema,
    'order_detail'                                      AS target_table,
    'parquet'                                           AS file_format,
    true                                                AS is_active,
    'full'                                              AS load_type
) AS src
ON  tgt.source_system = src.source_system
AND tgt.target_table  = src.target_table
WHEN NOT MATCHED THEN INSERT *
"""

spark.sql(merge_sql)
print("Seed-rijen geladen (MERGE — geen duplicaten bij herdraaien).")

## Stap 6 — Validatie & row counts

In [0]:
result = spark.sql(
    f"SELECT * FROM {catalog}.CONFIG.pipeline_sources ORDER BY target_table"
)
print(f"\nAantal rijen in control table: {result.count()}")
result.show(truncate=False)

## Resultaat

Setup voltooid. De volgende objecten zijn aangemaakt (of bestonden al):

| Object | Pad |
|--------|-----|
| Catalog | `DEMO` |
| Schema | `DEMO.STAGING_AZURESTORAGE` |
| Schema | `DEMO.STAGING_SQLSERVER` |
| Schema | `DEMO.CONFIG` |
| External Volume | `DEMO.STAGING_AZURESTORAGE.parquet` |
| Control table | `DEMO.CONFIG.pipeline_sources` (2 rijen) |